# Linear modeling for differential ADT expression

In this notebook, we retrieve our CD4 and CD8 T cells and our cell type labels, then perform differential expression of ADT data using linear models. Comparisons will be carried out for untreaded cells (DMSO-only controls) relative to T0 for each timepoint within each cell type.

To balance cell counts, we'll group cells by later timepoint or T0 and cell type, then use the minimum number of cells across all samples. For example, to test CD4 Naive T cells at T24, we'll examine the number of CD4 Naive cells in untreated conditions at 0, 4, 24, and 72 hours and randomly sample based on the minimum counts from all 4 samples.

We'll then perform comparisons between time points and T0 at each timepoint (e.g. CD4 Naive w/DMSO @ 4 hr vs. CD4 Naive w/DMSO @0 hr).

## Load packages

hise: The Human Immune System Explorer R SDK package  
purrr: Functional programming tools  
dplyr: Dataframe handling functions  
Seurat: single cell genomics methods  

In [1]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }
quiet_library(hise)
quiet_library(purrr)
quiet_library(dplyr)
quiet_library(Seurat)

Warning message:
“package ‘purrr’ was built under R version 4.4.3”
Warning message:
“package ‘dplyr’ was built under R version 4.4.3”
Warning message:
“package ‘Seurat’ was built under R version 4.4.3”
Warning message:
“package ‘SeuratObject’ was built under R version 4.4.3”
Warning message:
“package ‘sp’ was built under R version 4.4.3”


## Retrieve files

Now, we'll use the HISE SDK package to retrieve the Seurat objects and cell type labels based on file UUIDs. This will be placed in the `cache/` subdirectory by default.

In [2]:
file_uuids <- list(
    "7bdac6ef-e5e5-4150-b4f3-9c1a1e250334", # CD4 T cell Seurat object
    "46438bc4-cde6-4ae6-b349-9c513dd9d16f", # CD8 T cell Seurat object
    "ebd4bee7-2f5d-46e1-b2fc-22157f1b8d04", # CD4 type labels
    "4d6aade9-288c-452f-8f0d-ac59e539f4cc"  # CD8 type labels
)

In [3]:
fres <- cacheFiles(file_uuids)

[2026-10-07 14:31:33] INFO  Calling cacheFiles
[1] "downloading fileID 7bdac6ef-e5e5-4150-b4f3-9c1a1e250334"
[1] "downloading fileID 46438bc4-cde6-4ae6-b349-9c513dd9d16f"
[1] "downloading fileID ebd4bee7-2f5d-46e1-b2fc-22157f1b8d04"
[1] "downloading fileID 4d6aade9-288c-452f-8f0d-ac59e539f4cc"
[2026-10-07 14:31:44] INFO  Finished cacheFiles successfully (time_elapsed=7.843s)


## Select cells


In [4]:
cd4_labels <- read.csv(fres[[3]])
cd8_labels <- read.csv(fres[[4]])

In [5]:
all_labels <- rbind(cd4_labels, cd8_labels)

In [6]:
head(all_labels)

,barcodes,treatment,timepoint,predicted.celltype.l1.score,predicted.celltype.l1,predicted.celltype.l2.score,predicted.celltype.l2,predicted.celltype.l3.score,predicted.celltype.l3,aifi_cell_type
,<chr>,<chr>,<int>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>
1,2da9d348fb8111eda35df29f570c0793,dmso,24,1,CD4 T,0.7379073,CD4 Naive,0.7379073,CD4 Naive,t_cd4_naive
2,2daec6d2fb8111eda35df29f570c0793,dmso,24,1,CD4 T,1.0000000,CD4 Naive,1.0000000,CD4 Naive,t_cd4_naive
3,2db119d2fb8111eda35df29f570c0793,dmso,24,1,CD4 T,0.6491493,CD4 TCM,0.4892181,CD4 TCM_1,t_cd4_naive
4,2db582c4fb8111eda35df29f570c0793,dmso,24,1,CD4 T,0.8972198,CD4 Naive,0.8972198,CD4 Naive,t_cd4_naive
5,2db6727efb8111eda35df29f570c0793,dmso,24,1,CD4 T,0.3939763,CD4 TCM,0.2974696,CD4 Naive,t_cd4_naive
6,2dc35a20fb8111eda35df29f570c0793,dmso,24,1,CD4 T,0.6306972,CD4 Naive,0.6306972,CD4 Naive,t_cd4_naive


Exclude untreated cells - we won't use these for our treatment comparisons

In [7]:
all_labels <- all_labels %>%
  filter(treatment %in% c("dmso","untreated"))

In [8]:
table(all_labels$treatment, all_labels$timepoint)

           
                0     4    24    72
  dmso          0 10819 22177  2920
  untreated 18809     0     0     0

Get counts of each cell type for each sample:

In [9]:
count_summary <- all_labels %>%
  group_by(treatment, timepoint, aifi_cell_type) %>%
  summarise(n_cells = n(),
            .groups = "keep") %>%
  ungroup()

In [10]:
head(count_summary)

treatment,timepoint,aifi_cell_type,n_cells
<chr>,<int>,<chr>,<int>
dmso,4,t_cd4_cm,2679
dmso,4,t_cd4_em,819
dmso,4,t_cd4_naive,4420
dmso,4,t_cd4_treg,567
dmso,4,t_cd8_memory,886
dmso,4,t_cd8_naive,1448


Add a column for untreated counts per type and timepoint

In [11]:
count_summary <- count_summary %>%
  ungroup() %>%
  group_by(aifi_cell_type) %>%
  mutate(n_untreated = n_cells[treatment == "untreated"]) %>%
  ungroup() %>%
  filter(treatment != "untreated")

Regroup by treatment and cell type, and use treatment and DMSO counts to find minimums for sampling

In [12]:
type_minimums <- count_summary %>%
  group_by(timepoint, aifi_cell_type) %>%
  mutate(n_sample = min(c(n_cells, n_untreated)))

In [13]:
type_minimums %>%
  arrange(treatment, aifi_cell_type, timepoint)

treatment,timepoint,aifi_cell_type,n_cells,n_untreated,n_sample
<chr>,<int>,<chr>,<int>,<int>,<int>
dmso,4,t_cd4_cm,2679,4614,2679
dmso,24,t_cd4_cm,5262,4614,4614
dmso,72,t_cd4_cm,648,4614,648
dmso,4,t_cd4_em,819,1481,819
dmso,24,t_cd4_em,1812,1481,1481
dmso,72,t_cd4_em,180,1481,180
dmso,4,t_cd4_naive,4420,6876,4420
dmso,24,t_cd4_naive,9299,6876,6876
dmso,72,t_cd4_naive,1352,6876,1352


In [14]:
comp_list <- map(
    1:nrow(type_minimums),
    function(i) {
        as.list(type_minimums[i,])
    }
)

## Sample cells for each test

Here, we'll sample cells for comparisons and generate a table of foreground and background cells to use for analysis.

In [15]:
sampled_comp_cells <- map(
    comp_list,
    function(comp) {
        set.seed(3030)
        
        tp <- comp$timepoint
        ct <- comp$aifi_cell_type
        
        fg_treat <- comp$treatment
        bg_treat <- "untreated"

        n_sample <- comp$n_sample

        fg_cells <- all_labels %>%
          filter(treatment == fg_treat,
                 timepoint == tp,
                 aifi_cell_type == ct) %>%
          sample_n(n_sample)
        bg_cells <- all_labels %>%
          filter(treatment == bg_treat,
                 aifi_cell_type == ct) %>%
          sample_n(n_sample)

        rbind(bg_cells, fg_cells)
    }
)

## Build matrices for each test

Now, we'll use the selected cells to build a data matrix for each comparison.

We'll use these together with the cell metadata to run `lm()`.

In [16]:
cd4_so <- readRDS(fres[[1]])
cd8_so <- readRDS(fres[[2]])

In [17]:
all_so <- merge(cd4_so, cd8_so)

In [18]:
DefaultAssay(all_so) <- "ADT"

In [19]:
all_so <- NormalizeData(
    all_so, 
    normalization.method = "CLR",
    margin = 2
)

Normalizing across cells



In [20]:
all_mat <- all_so[["ADT"]]@data
rm(cd4_so)
rm(cd8_so)
rm(all_so)

In [21]:
sampled_comp_mats <- map(
    sampled_comp_cells,
    function(meta) {
        all_mat[,meta$barcodes]
    }
)

In [22]:
head(sampled_comp_cells[[1]])

,barcodes,treatment,timepoint,predicted.celltype.l1.score,predicted.celltype.l1,predicted.celltype.l2.score,predicted.celltype.l2,predicted.celltype.l3.score,predicted.celltype.l3,aifi_cell_type
,<chr>,<chr>,<int>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>
1,418831ecfb8211ed9d09f6494964bb81,untreated,0,1,CD4 T,1.0000000,CD4 TCM,1.0000000,CD4 TCM_3,t_cd4_cm
2,8fe39300fb8111ed99031af2246d8fa2,untreated,0,1,CD4 T,0.7577974,CD4 TCM,0.6568277,CD4 TCM_2,t_cd4_cm
3,44f8a08cfb8211ed9d09f6494964bb81,untreated,0,1,CD4 T,0.9146994,CD4 TCM,0.4444861,CD4 TCM_3,t_cd4_cm
4,757fd402fb8011eda0a9e2763843e521,untreated,0,1,CD4 T,0.7770264,CD4 TCM,0.4867559,CD4 TCM_2,t_cd4_cm
5,f7f4d500fb7f11ed898e2a71e2203848,untreated,0,1,CD4 T,0.9788960,CD4 TCM,0.5849661,CD4 TCM_1,t_cd4_cm
6,40e792f0fb8211ed9d09f6494964bb81,untreated,0,1,CD4 T,0.5855544,CD4 TCM,0.3596720,CD4 TCM_1,t_cd4_cm


In [23]:
lm_res <- map2_dfr(
    sampled_comp_cells,
    sampled_comp_mats,
    function(meta, mat) {
        set.seed(3030)
        
        treatments <- unique(meta$treatment)
        fg_treat <- treatments[treatments != "untreated"]
        treat_levels <- c("untreated", fg_treat)
        meta$treatment <- factor(meta$treatment, levels = treat_levels)
        
        ct <- meta$aifi_cell_type[1]
        tp <- meta$timepoint[meta$treatment == fg_treat][1]
        fg <- fg_treat
        bg <- "untreated"
        
        ds <- nrow(meta) / 2
        
        map_dfr(rownames(mat),
            function(feat) {
                dat <- data.frame(
                    treatment = meta$treatment, 
                    val = mat[feat,]
                )
                names(dat)[2] <- feat
                
                lm_res <- lm(
                    formula = as.formula(paste0("`",feat,"` ~ treatment")), 
                    data = dat)
                
                coef <- summary(lm_res)$coefficients
                
                data.frame(
                    aifi_cell_type = ct,
                    timepoint = tp,
                    fg = fg,
                    bg = bg,
                    n_downsample = ds,
                    feature = feat,
                    estimate = coef[2,1],
                    std_error = coef[2,2],
                    t_value = coef[2,3],
                    nomP = coef[2,4],
                    mean = mean(dat[[feat]]),
                    fg_mean = mean(dat[[feat]][dat$treatment == fg]),
                    bg_mean = mean(dat[[feat]][dat$treatment != fg]),
                    logFC = mean(dat[[feat]][dat$treatment == fg]) - mean(dat[[feat]][dat$treatment != fg])
                )
        })
    }
)

In [24]:
lm_res$adjP <- p.adjust(lm_res$nomP, method = "BH")

In [25]:
table(lm_res$timepoint)


  4  24  72 
330 330 330 

In [26]:
table(lm_res$timepoint, lm_res$aifi_cell_type)

    
     t_cd4_cm t_cd4_em t_cd4_naive t_cd4_treg t_cd8_memory t_cd8_naive
  4        55       55          55         55           55          55
  24       55       55          55         55           55          55
  72       55       55          55         55           55          55

In [27]:
lm_res %>%
  filter(feature == "CD69") %>%
  arrange(timepoint, aifi_cell_type)

aifi_cell_type,timepoint,fg,bg,n_downsample,feature,estimate,std_error,t_value,nomP,mean,fg_mean,bg_mean,logFC,adjP
<chr>,<int>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
t_cd4_cm,4,dmso,untreated,2679,CD69,1.44324543,0.014876034,97.018160,0.000000e+00,1.2271422,1.9487649,0.5055195,1.44324543,0.000000e+00
t_cd4_em,4,dmso,untreated,819,CD69,1.40239769,0.024436579,57.389282,0.000000e+00,1.2226262,1.9238250,0.5214273,1.40239769,0.000000e+00
t_cd4_naive,4,dmso,untreated,4420,CD69,1.08642854,0.010345795,105.011606,0.000000e+00,1.0078413,1.5510556,0.4646270,1.08642854,0.000000e+00
t_cd4_treg,4,dmso,untreated,567,CD69,0.81511839,0.025941704,31.421159,2.476069e-156,0.8570247,1.2645839,0.4494655,0.81511839,9.805232e-155
t_cd8_memory,4,dmso,untreated,886,CD69,1.26037869,0.029516358,42.701023,1.825162e-274,1.1652276,1.7954169,0.5350382,1.26037869,1.062888e-272
t_cd8_naive,4,dmso,untreated,1448,CD69,1.08791944,0.019979712,54.451207,0.000000e+00,1.0002492,1.5442089,0.4562895,1.08791944,0.000000e+00
t_cd4_cm,24,dmso,untreated,4614,CD69,0.12507672,0.007665500,16.316838,5.002708e-59,0.5669202,0.6294585,0.5043818,0.12507672,9.711139e-58
t_cd4_em,24,dmso,untreated,1481,CD69,0.19823401,0.015817422,12.532637,3.800315e-35,0.6147028,0.7138198,0.5155858,0.19823401,5.452625e-34
t_cd4_naive,24,dmso,untreated,6876,CD69,0.02008337,0.004096663,4.902374,9.576611e-07,0.4759863,0.4860280,0.4659446,0.02008337,5.544354e-06


## Generate output files

For downstream use, we'll output the table of DEP results.

In [28]:
dir.create("output")

Warning message in dir.create("output"):
“'output' already exists”


In [29]:
out_res <- paste0("output/untreated_lm_dep_",Sys.Date(),".csv")
write.csv(lm_res,
          out_res,
          quote = FALSE, row.names = FALSE)

## Store results in HISE

Finally, we store the output file in our Collaboration Space for later retrieval and use. We need to provide the UUID for our Collaboration Space (aka `studySpaceId`), as well as a title for this step in our analysis process.

The hise function `uploadFiles()` also requires the FileIDs from the original fileset for reference.

In [30]:
study_space_uuid <- "40df6403-29f0-4b45-ab7d-f46d420c422e"
title <- paste("VRd TEA-seq lm DEP with means for untreated", Sys.Date(), "3")

In [31]:
set.seed(NULL)
search_id <- ids::adjective_animal()
search_id

[1] "rightable_australiankelpie"

In [32]:
out_list <- list(out_res)

In [33]:
out_list

[[1]]
[1] "output/untreated_lm_dep_2026-10-07.csv"

In [34]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/archrpixiv12/.pixi/envs/default/lib/libopenblasp-r0.3.33.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] Seurat_5.5.0       SeuratObject_5.4.0 sp_2.2-1           dplyr_1.2.1       
[5] purrr_1.2.2        hise_3.6.15       

loaded via a namespace (and not attached):
  [1] bitops_1.0-9           deldir_2.0-4           pbapply_1.7-4         
  [4] gridExtra_2.3          rlang_1.2.0            magr

In [35]:
uploadFiles(
    files = out_list,
    studySpaceId = study_space_uuid,
    title = title,
    inputFileIds = file_uuids,
    destination = search_id
)

Please provide input of comma separated sample ids for the files being uploaded. If you do not have any sample ids, press enter:  


[2026-10-07 14:32:47] INFO  Calling uploadFiles
[2026-10-07 14:33:32] INFO  Finished uploadFiles successfully (time_elapsed=42.633s)


$Message
[1] "General Okay-ness"

$VisualizationId
[1] "00000000-0000-0000-0000-000000000000"

$AbstractionId
[1] "00000000-0000-0000-0000-000000000000"

$TraceId
[1] "75255d82-7893-48d3-86fc-24cf24439204"

$ProcessId
[1] "dd60f5cc-a24c-41a7-9916-a780409f44fe"

$WorkflowId
[1] "11c669db-3150-4261-b7b7-59fb02497d0d"

$FileIds
$FileIds[[1]]
[1] "a20a208b-f1b5-401d-a349-7d3ea22514ac"